# 3. Bootstrapping a discount curve

**Level:** L1. **Tags:** `rates`, `curve`, `bootstrap`.
**Library:** `fi_rates.bootstrap`, `fi_rates.curve` (`python/equity/04-fixed-income`).

**Plain idea.** Each market quote pins one more point of the discount
curve. Short deposits give the front directly; each par swap then adds one
pillar so that the swap is worth zero.

**Formal statement** (Hagan and West, "Interpolation methods for curve
construction", *Applied Mathematical Finance* 13, 2006): a deposit gives
$P(T) = 1/(1 + rT)$; a par swap with annual fixed leg satisfies
$r\sum_i P(t_i) + P(T) = 1$.

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
from fi_rates.bootstrap import (  # noqa: F401
    Deposit,
    ParSwap,
    bootstrap_curve,
    reprice_instruments,
)

## Task 1: build the curve

Quotes: deposits at 0.25y 3.00%, 0.5y 3.10%, 1y 3.20%; annual par swaps at
2y 3.35%, 3y 3.45%, 5y 3.60%, 7y 3.70%, 10y 3.80%. Bootstrap with the
default log-linear discount-factor interpolation. Store it as `curve`.

In [ ]:
quotes = [...]  # Deposit(maturity, rate) and ParSwap(maturity, rate)
curve = ...
# Exercise: replace the ... above, then run the check cell below.

**Check 1: every quote reprices.** This is the definition of a bootstrap.

In [ ]:
errors = [err for _, err in reprice_instruments(quotes, curve)]
assert max(abs(e) for e in errors) < 1e-10

**Check 2: by hand.** The 1y deposit gives $P(1) = 1/1.032$ exactly. The 2y
swap then needs only $P(1)$ and $P(2)$ (annual fixed leg), so
$P(2) = (1 - 0.0335 P(1)) / 1.0335$. Neither needs interpolation.

In [ ]:
p1 = 1 / 1.032
p2 = (1 - 0.0335 * p1) / 1.0335
assert abs(float(curve.df(1.0)) - p1) < 1e-12
assert abs(float(curve.df(2.0)) - p2) < 1e-12
assert abs(curve.par_rate(5.0) - 0.0360) < 1e-10

## Task 2: forwards and a parallel shift

Compute the 1y-into-1y simple forward, and the DV01 of a 5y annual swap
paying fixed at 3.60% on 1,000,000 notional, using a 1bp parallel bump.

In [ ]:
fwd_1y1y = ...
dv01 = ...  # PV change of a 5y payer swap for +1bp
# Exercise: replace the ... above, then run the check cell below.

In [ ]:
assert abs(fwd_1y1y - (p1 / p2 - 1)) < 1e-12
annuity = sum(float(curve.df(t)) for t in range(1, 6))
# A payer gains about notional x annuity x 1bp when rates rise.
assert abs(dv01 / (1_000_000 * annuity * 1e-4) - 1) < 0.05

**Takeaways.** Reprice the inputs, check the first pillars by hand, and
check a risk number against the annuity rule of thumb.